[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/calculus/c02_zoom.ipynb)

# Can a curve have no slope anywhere? The zoom test

In 1893 Charles Hermite wrote that he turned away "with fright and horror" from continuous functions with no
derivatives. Here you hold the lens: zoom into a smooth curve, a corner, a jump and Weierstrass's curve, and see
which ones straighten out.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the
▶ Run button) to run it. Then run each cell in turn, from top to bottom. When a slider appears, drag it. Change any
number you like and run the cell again: nothing can break. It works in your browser (no install) and in Google Colab.

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look: one style for every figure in these notebooks. Run this cell first.
import math, warnings
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatLogSlider, ToggleButtons

SLATE, SLATE2 = "#1f2b28", "#2a3a36"        # the slate ground
CHALK, DIM = "#ece6d6", "#93a097"           # chalk, and chalk smudge
CORAL = "#e3896b"                           # the one key thing in each figure
TEAL, SAGE, LILAC = "#6fb6bb", "#a0c27c", "#c9a0d3"   # the curve · slopes and rates · areas

warnings.filterwarnings("ignore", message=".*as float was deprecated.*")   # a browser-only notice, harmless
plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#3b4c47", "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "legend.facecolor": SLATE2, "legend.edgecolor": DIM, "legend.labelcolor": CHALK, "legend.framealpha": 0.9,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

def weierstrass(x, a=0.5, b=3, terms=24):
    """Weierstrass's curve: a sum of cosine waves, each a times as tall and b times as fast as the one before."""
    x = np.asarray(x, dtype=float)
    return sum(a ** n * np.cos(b ** n * np.pi * x) for n in range(terms))

def window_slope(f, a, h):
    """The slope of the window from a to a + h (h can be negative: a window to the left)."""
    return (f(a + h) - f(a)) / h

CURVES = {   # name: (function, the point we zoom in on)
    "smooth: y = x²": (lambda x: x * x, 1.0),
    "the V: y = |x|": (lambda x: np.abs(x), 0.0),
    "the jump: parcel price": (lambda x: np.where(x <= 1, 2.0, 4.0), 1.0),
    "Weierstrass's curve": (weierstrass, 0.0),
}

## 1 · Zoom in

Pick a curve with the buttons, then drag the zoom slider. The view always stays centred on the coral point, and its
height is scaled to fit what is inside it. A **differentiable** curve straightens out as you zoom in: close up, it
looks like a line with one slope.

In [ ]:
def zoom(curve="smooth: y = x²", zoom=1.0):
    f, a = CURVES[curve]
    half = 1.0 / zoom
    x = np.linspace(a - half, a + half, 2001)
    y = f(x)
    fig, ax = plt.subplots(figsize=(10, 5.2))
    ax.plot(x, y, color=TEAL, lw=2.6)
    ax.plot([a], [f(np.array(a))], "o", color=CORAL, ms=10, zorder=5)
    lo, hi = float(np.min(y)), float(np.max(y))
    pad = 0.08 * (hi - lo if hi > lo else 1.0)
    ax.set_ylim(lo - pad, hi + pad); ax.set_xlim(a - half, a + half)
    ax.set_xlabel("x"); ax.ticklabel_format(useOffset=False)
    ax.set_title(f"{curve}   ·   zoom ×{zoom:,.0f}")
    plt.show()

interact(zoom, curve=ToggleButtons(options=list(CURVES), value="smooth: y = x²", description="curve"),
         zoom=FloatLogSlider(value=1.0, base=10, min=0, max=3, step=0.25, description="zoom ×", readout_format=".0f"));

**Notice.** The parabola straightens. The V stays the same V at every zoom: its corner never goes away. The jump
stays a jump. And Weierstrass's curve stays jagged, however far you zoom in.

## 2 · Walking in and walking out

At a point, take windows on the right (walking out) and on the left (walking in), and shrink them. The curve has a
slope there only if both sides head to the same number.

In [ ]:
for name, (f, a) in CURVES.items():
    print(name, f"at x = {a:g}")
    for k in range(1, 5):
        h = 10.0 ** -k
        print(f"   width {h:<7} right {float(window_slope(f, a, h)):12.3f}    left {float(window_slope(f, a, -h)):12.3f}")

**Notice.** For x² both sides head to 2. For the V, the right side says 1 and the left side says −1, for every
window. For the jump, the right side runs away (20, 200, 2000…). For Weierstrass's curve the numbers never settle.

## 3 · Build Weierstrass's curve, wave by wave

Start with one cosine wave. Each new wave is half as tall and three times as fast. Drag the slider to add waves.
Every wave is smooth on its own; the wiggles pile up.

In [ ]:
from ipywidgets import IntSlider

def build(waves=1):
    x = np.linspace(-1, 1, 4001)
    fig, ax = plt.subplots(figsize=(10, 4.6))
    ax.plot(x, weierstrass(x, terms=waves), color=TEAL, lw=1.8)
    ax.set_title(f"{waves} wave{'s' if waves > 1 else ''}: the newest is {1.5 ** (waves - 1):.1f}× as steep as the first")
    ax.set_xlabel("x"); ax.set_ylim(-2.2, 2.2)
    plt.show()

interact(build, waves=IntSlider(value=1, min=1, max=12, description="waves"));

## 4 · The textbook join

f(x) = x² + k for x ≤ 2, and 3x for x > 2. The two pieces meet when 4 + k = 6, so k = 2. Then check the slopes from
both sides.

In [ ]:
k = 2
f = lambda x: np.where(x <= 2, x * x + k, 3 * x)
for w in (0.1, 0.01, 0.001):
    print(f"width {w:<6} left slope {window_slope(f, 2.0, -w):.4f}   right slope {window_slope(f, 2.0, w):.4f}")
x = np.linspace(0.5, 3.5, 400)
fig, ax = plt.subplots(figsize=(8, 4.4))
ax.plot(x, f(x), color=TEAL, lw=3)
ax.plot([2], [6], "o", color=CORAL, ms=10)
ax.set_title("continuous at 2 (k = 2), but the slopes are 4 and 3: a corner"); ax.set_xlabel("x")
plt.show()

**Try it.** Change `k` to 1 and run the cell again: the pieces no longer meet. Then try the challenge from the
video: is x|x| differentiable at 0? Change `f` to `lambda x: x * np.abs(x)` and look at both sides at 0.

## Sources
Hermite to Stieltjes, 20 May 1893 (Correspondance d'Hermite et de Stieltjes, 1905). K. Weierstrass, lecture to the
Berlin Academy, 18 July 1872. G. H. Hardy (1916) on a b ≥ 1. MacTutor History of Mathematics. Retrieved 2026-10-24.